# DeepFilterNet + Resemble Video Audio Replacement

This Colab notebook runs the selected best workflow from the experiment notebook:

1. Extract the original video audio with FFmpeg.
2. Run DeepFilterNet -> Resemble Enhance.
3. Convert the enhanced audio to a 48 kHz comparison WAV.
4. Create a new MP4 with the original video stream and the enhanced replacement audio.

Use a GPU runtime when possible: `Runtime > Change runtime type > T4 GPU`.


## 1. Install Runtime Dependencies

Run this cell first. The model install can take a few minutes on a fresh Colab runtime.


In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg wget
%pip install -q --upgrade pip
%pip install -q git+https://github.com/resemble-ai/resemble-enhance.git soundfile pandas


## 2. Imports, Paths, and Settings

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import Audio, Video, display, HTML

ROOT = Path("/content/deepfilter_resemble_video")
INPUT_DIR = ROOT / "input"
WORK_DIR = ROOT / "work"
OUTPUT_DIR = ROOT / "outputs"
LOG_DIR = ROOT / "logs"
BIN_DIR = ROOT / "bin"

for directory in [INPUT_DIR, WORK_DIR, OUTPUT_DIR, LOG_DIR, BIN_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

VIDEO_PATH = None
BASELINE_WAV = WORK_DIR / "baseline_48k.wav"
FINAL_VIDEO = OUTPUT_DIR / "deepfilter_resemble_enhanced_video.mp4"
COMPARISON_WAV = OUTPUT_DIR / "deepfilter_resemble_comparison_48k.wav"


## 3. Upload or Mount the Input Video

Option A uploads a local video into the Colab session. Option B copies a video from Google Drive.


In [ ]:
USE_GOOGLE_DRIVE = False
DRIVE_VIDEO_PATH = ""  # Example: "/content/drive/MyDrive/input_video.mp4"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    source_video = Path(DRIVE_VIDEO_PATH)
    if not source_video.exists():
        raise FileNotFoundError(f"Drive video not found: {source_video}")
    target_video = INPUT_DIR / source_video.name
    shutil.copy2(source_video, target_video)
else:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No video uploaded.")
    uploaded_name = next(iter(uploaded.keys()))
    uploaded_path = Path(uploaded_name)
    target_video = INPUT_DIR / uploaded_path.name
    shutil.move(str(uploaded_path), target_video)

VIDEO_PATH = target_video
print("Input video:", VIDEO_PATH)
display(Video(filename=str(VIDEO_PATH), embed=True))


## 4. Shared FFmpeg and Filesystem Helpers

In [ ]:
def run_cmd(cmd, log_name=None, check=True):
    cmd = [str(part) for part in cmd]
    print("+", " ".join(cmd))
    proc = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if log_name:
        (LOG_DIR / log_name).write_text(proc.stdout, encoding="utf-8")
    if proc.returncode != 0 and check:
        raise RuntimeError(proc.stdout)
    return proc.stdout


def safe_reset_dir(path):
    path = Path(path)
    resolved = path.resolve()
    root_resolved = ROOT.resolve()
    if resolved != root_resolved and root_resolved not in resolved.parents:
        raise ValueError(f"Refusing to reset directory outside {ROOT}: {path}")
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def existing_wavs(directory):
    directory = Path(directory)
    if not directory.exists():
        return set()
    return {p.resolve() for p in directory.rglob("*.wav")}


def newest_new_wav(directory, before):
    after = existing_wavs(directory)
    new_files = list(after - set(before))
    if not new_files:
        candidates = list(Path(directory).rglob("*.wav"))
        if not candidates:
            raise FileNotFoundError(f"No WAV output found under {directory}")
        new_files = candidates
    return max(new_files, key=lambda p: p.stat().st_mtime)


def extract_audio(video_path, output_wav=BASELINE_WAV):
    run_cmd([
        "ffmpeg", "-y", "-i", video_path,
        "-vn", "-ac", "1", "-ar", "48000", "-sample_fmt", "s16",
        output_wav,
    ], "extract_audio.log")
    return Path(output_wav)


def make_comparison_wav(input_wav, output_wav):
    run_cmd([
        "ffmpeg", "-y", "-i", input_wav,
        "-ac", "1", "-ar", "48000", "-sample_fmt", "s16",
        output_wav,
    ], f"{Path(output_wav).stem}_comparison.log")
    return Path(output_wav)


def remux_video(video_path, audio_wav, output_mp4):
    run_cmd([
        "ffmpeg", "-y", "-i", video_path, "-i", audio_wav,
        "-map", "0:v:0", "-map", "1:a:0",
        "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
        "-shortest", output_mp4,
    ], f"{Path(output_mp4).stem}_remux.log")
    return Path(output_mp4)


## 5. DeepFilterNet Binary Setup

The Python `deepfilternet` package can fail on Colab when its Rust-backed dependency has no compatible wheel. This notebook uses the upstream Linux `deep-filter` release binary, matching the experiment notebook.


In [ ]:
DEEP_FILTER_VERSION = "0.5.6"
DEEP_FILTER_BINARY_URL = "https://github.com/Rikorose/DeepFilterNet/releases/download/v0.5.6/deep-filter-0.5.6-x86_64-unknown-linux-musl"
DEEP_FILTER_BIN = BIN_DIR / "deep-filter"


def install_deepfilter_binary():
    if DEEP_FILTER_BIN.exists():
        print("DeepFilterNet binary already exists:", DEEP_FILTER_BIN)
    else:
        run_cmd(
            ["wget", "-q", "-O", DEEP_FILTER_BIN, DEEP_FILTER_BINARY_URL],
            "install_deepfilter_binary.log",
        )
        DEEP_FILTER_BIN.chmod(0o755)

    version_output = run_cmd(
        [DEEP_FILTER_BIN, "--version"],
        "deepfilter_binary_version.log",
        check=False,
    )
    if version_output.strip():
        print(version_output)
    return DEEP_FILTER_BIN


install_deepfilter_binary()


## 6. Inspect Media and Extract Baseline WAV

The baseline extraction is mono, 48 kHz, 16-bit PCM so the enhancement chain has a stable input.


In [ ]:
if VIDEO_PATH is None or not Path(VIDEO_PATH).exists():
    raise RuntimeError("Run the upload/mount cell first.")

probe_output = run_cmd(["ffprobe", "-hide_banner", "-i", VIDEO_PATH], "ffprobe_input.log", check=False)
print(probe_output)

BASELINE_WAV = extract_audio(VIDEO_PATH)
print("Baseline WAV:", BASELINE_WAV)
display(Audio(filename=str(BASELINE_WAV)))


## 7. Model Runner Functions

In [ ]:
def run_deepfilternet(input_wav, out_dir):
    if "DEEP_FILTER_BIN" not in globals():
        raise RuntimeError("Run the DeepFilterNet binary setup cell before processing.")

    if not Path(DEEP_FILTER_BIN).exists():
        install_deepfilter_binary()

    out_dir = safe_reset_dir(out_dir)
    before = existing_wavs(out_dir)
    run_cmd([DEEP_FILTER_BIN, "--output-dir", out_dir, input_wav], "deepfilternet.log")
    return newest_new_wav(out_dir, before)


def get_resemble_command():
    import importlib.util

    for executable_name in ["resemble-enhance", "resemble_enhance"]:
        cli_path = shutil.which(executable_name)
        if cli_path:
            print("Using Resemble Enhance CLI:", cli_path)
            return [cli_path]

    if importlib.util.find_spec("resemble_enhance") is not None:
        print("Resemble Enhance CLI was not found on PATH; using python module fallback.")
        return [sys.executable, "-m", "resemble_enhance.enhancer.__main__"]

    raise RuntimeError(
        "Resemble Enhance is not importable. Restart the Colab runtime, run the install cell, "
        "and confirm it installs git+https://github.com/resemble-ai/resemble-enhance.git."
    )


def run_resemble(input_wav, out_dir):
    out_dir = safe_reset_dir(out_dir)
    in_dir = out_dir / "input"
    result_dir = out_dir / "result"
    in_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)

    copied_input = in_dir / Path(input_wav).name
    shutil.copy2(input_wav, copied_input)

    before = existing_wavs(result_dir)
    cmd = get_resemble_command() + [in_dir, result_dir]
    run_cmd(cmd, "resemble_enhance.log")
    return newest_new_wav(result_dir, before)


## 8. Run DeepFilterNet -> Resemble Enhance and Remux Video

This cell replaces the original audio track by mapping only the original video stream and the enhanced audio stream into the final MP4.


In [ ]:
if not Path(BASELINE_WAV).exists():
    raise RuntimeError("Run the baseline extraction cell before processing.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

deepfilter_wav = run_deepfilternet(BASELINE_WAV, WORK_DIR / "deepfilternet")
print("DeepFilterNet WAV:", deepfilter_wav)

def run_selected_workflow():
    resemble_wav = run_resemble(deepfilter_wav, WORK_DIR / "resemble")
    comparison_wav = make_comparison_wav(
        resemble_wav,
        OUTPUT_DIR / "deepfilter_resemble_comparison_48k.wav",
    )
    final_video = remux_video(
        VIDEO_PATH,
        comparison_wav,
        OUTPUT_DIR / "deepfilter_resemble_enhanced_video.mp4",
    )
    return resemble_wav, comparison_wav, final_video

resemble_wav, comparison_wav, FINAL_VIDEO = run_selected_workflow()

result = {
    "workflow": "deepfilternet_resemble_video_remux",
    "source_video": str(VIDEO_PATH),
    "deepfilter_wav": str(deepfilter_wav),
    "resemble_wav": str(resemble_wav),
    "comparison_wav": str(comparison_wav),
    "enhanced_video": str(FINAL_VIDEO),
}

display(pd.DataFrame([result]))


## 9. Preview Enhanced Audio and Video

In [ ]:
if not Path(comparison_wav).exists() or not Path(FINAL_VIDEO).exists():
    raise RuntimeError("Run the processing cell before previewing outputs.")

display(HTML("<h3>Enhanced replacement audio</h3>"))
display(Audio(filename=str(comparison_wav)))

display(HTML("<h3>Final video with enhanced audio</h3>"))
display(Video(filename=str(FINAL_VIDEO), embed=True))


## 10. Archive and Download Outputs

In [ ]:
archive_path = shutil.make_archive("/content/deepfilter_resemble_video_outputs", "zip", ROOT)
print("Archive created:", archive_path)

try:
    from google.colab import files
    files.download(archive_path)
except Exception as exc:
    print("Automatic download did not start. Use the Files sidebar to download:")
    print(archive_path)
    print("Download error:", exc)
